# ۱ · تشخیص و جستجوی چهره

جستجوی بیومتریک روی آرشیو: یک عکس از یک نفر می‌دهی، همه‌ی عکس‌هایی که
آن فرد در آن‌هاست را مرتب‌شده بر اساس شباهت برمی‌گرداند.

**مدل:** InsightFace `buffalo_l` = RetinaFace (تشخیص) + ArcFace (بازشناسی)

---

## 🐞 باگ‌هایی که رفع شد

| # | مشکل | اثر | راه‌حل |
|:--|:--|:--|:--|
| ۱ | جدول `face_embeddings` کلید اصلی نداشت | ایندکس مجدد ردیف تکراری انباشته می‌کرد | `PRIMARY KEY (file_name, bbox)` |
| ۲ | ذخیره با `pickle` | ریسک اجرای کد + حجم چند برابر | `float32` خام |
| ۳ | `break` هر تصویر گم شده بود | عکس با دو چهره‌ی منطبق دو بار در نتایج | `best_per_image` |
| ۴ | ThreadPool و checkpoint حذف شده بود | ایندکس نصفه‌کاره = از دست رفتن کل کار | برگردانده شد |

## ✅ چیزی که درست بود و نگه داشتیم

- **`normed_embedding`** به‌جای `embedding` خام. چون از قبل L2-نرمال است،
  ضرب داخلی **دقیقاً** کسینوس می‌شود. خیلی‌ها این را اشتباه می‌کنند و
  بعد متریک‌شان غلط درمی‌آید.
- انتخاب تطبیقی `det_size` + fallback به ۶۴۰ اگر چیزی پیدا نشد
- انتخاب بزرگ‌ترین چهره به‌عنوان سوژه‌ی عکس پرس‌وجو

In [ ]:
# ── نصب وابستگی‌ها ───────────────────────────────────────────────────
# ترتیب مهم است: insightface نسخه CPU از onnxruntime نصب می‌کند،
# پس اول آن را حذف و بعد نسخه GPU را نصب می‌کنیم.
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q insightface ultralytics transformers opencv-python tqdm
!pip uninstall -y onnxruntime -q
!pip install -q onnxruntime-gpu

import onnxruntime as ort
print("✅ نصب کامل شد")
print("موتورهای در دسترس:", ort.get_available_providers())

In [ ]:
# ── کش مدل‌ها روی Google Drive ───────────────────────────────────────
#
# مشکل: هر بار Colab باز می‌شود، مدل‌ها (چند گیگابایت) دوباره دانلود
# می‌شوند — هم کند، هم مصرف اینترنت.
#
# راه‌حل: یک بار دانلود، ذخیره در Drive، دفعات بعد مستقیم از Drive.
# فقط کش HuggingFace و torch.hub را به یک پوشه‌ی ماندگار می‌بریم.

import os

MODEL_CACHE = "/content/drive/MyDrive/vie_models"   # پوشه‌ی دلخواه در Drive

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    MODEL_CACHE = os.path.abspath("./vie_models")    # خارج از Colab

os.makedirs(f"{MODEL_CACHE}/huggingface", exist_ok=True)
os.makedirs(f"{MODEL_CACHE}/torch_hub", exist_ok=True)

os.environ["HF_HOME"] = f"{MODEL_CACHE}/huggingface"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{MODEL_CACHE}/huggingface/hub"
os.environ["TORCH_HOME"] = f"{MODEL_CACHE}/torch_hub"

print("✅ کش مدل‌ها:", MODEL_CACHE)
print("   بار اول دانلود، دفعات بعد مستقیم از Drive.")

In [ ]:
# ── تشخیص دستگاه و دقت عددی ─────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    کد قبلی سه جور مقایسه داشت:
#        if device.type == "cuda":   ← درست
#        if device == "cuda":        ← همیشه False !
#
#    چون device یک شیء torch.device است، نه رشته.
#    تست شده روی torch 2.8:  torch.device('cuda') == 'cuda'  →  False
#
#    نتیجه: روی GPU مدل half می‌شد ولی ورودی float32 می‌ماند →
#    RuntimeError: expected scalar type Half but found Float
#
# ✅ راه‌حل: دستگاه و dtype را یکجا حل می‌کنیم تا نتوانند با هم اختلاف پیدا کنند.

import torch
from dataclasses import dataclass


@dataclass(frozen=True)
class Runtime:
    device: torch.device
    use_half: bool

    @property
    def is_cuda(self) -> bool:
        return self.device.type == "cuda"

    def cast_inputs(self, inputs: dict) -> dict:
        """انتقال ورودی به دستگاه با dtype هماهنگ.
        فقط تنسورهای اعشاری half می‌شوند؛ input_ids باید عدد صحیح بماند."""
        out = {}
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                v = v.to(self.device)
                if self.use_half and v.is_floating_point():
                    v = v.half()
            out[k] = v
        return out

    def prepare_model(self, model):
        model = model.to(self.device)
        if self.use_half:
            model = model.half()
        return model.eval()


def resolve_runtime(preference: str = "auto", half: bool = True) -> Runtime:
    name = preference
    if preference == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(name)
    # fp16 روی CPU کندتر از fp32 است و بعضی عملیات پشتیبانی نمی‌شوند
    return Runtime(device=device, use_half=half and device.type == "cuda")


RT = resolve_runtime()
print(f"🚀 دستگاه: {RT.device}   |   نیمه‌دقت (FP16): {RT.use_half}")

In [ ]:
# ── ذخیره‌سازی امبدینگ ──────────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    کد قبلی از pickle استفاده می‌کرد:
#        pickle.dumps(face.normed_embedding.tolist())
#
#    دو مشکل:
#    ۱) امنیتی: pickle.loads روی فایلی که خودت نساخته‌ای =
#       اجرای کد دلخواه. فایل ایندکس دقیقاً همان چیزی است که
#       بین سیستم‌ها کپی می‌شود.
#    ۲) حجم: .tolist() یک لیست پایتون می‌سازد، نه آرایه —
#       چند برابر بایت خام.
#
# ✅ راه‌حل: float32 خام با tobytes/frombuffer

import numpy as np

DTYPE = np.dtype(np.float32).newbyteorder("<")   # little-endian، قابل حمل


def to_blob(vector) -> bytes:
    arr = np.asarray(vector, dtype=np.float32).ravel()
    if arr.size == 0:
        raise ValueError("امبدینگ خالی")
    if not np.all(np.isfinite(arr)):
        # یک NaN تمام شباهت‌هایی که در آن شرکت کند را مسموم می‌کند
        raise ValueError("امبدینگ شامل NaN یا بی‌نهایت است")
    return arr.astype(DTYPE, copy=False).tobytes()


def from_blob(blob: bytes) -> np.ndarray:
    if not blob:
        raise ValueError("بلاب خالی")
    if len(blob) % DTYPE.itemsize:
        raise ValueError("طول بلاب نادرست — شاید ایندکس با نسخه pickle قدیمی ساخته شده")
    return np.frombuffer(blob, dtype=DTYPE).astype(np.float32, copy=True)


# تست سریع
_v = np.random.randn(512).astype(np.float32)
assert np.allclose(from_blob(to_blob(_v)), _v, atol=1e-6)
print(f"✅ ذخیره‌سازی float32 سالم است — یک بردار ۵۱۲ بعدی = {len(to_blob(_v)):,} بایت")

In [ ]:
# ── امتیازدهی رقابتی ────────────────────────────────────────────────
#
# 💡 بهترین ایده‌ی پروژه — حالا در هر سه مسیر تشخیص استفاده می‌شود.
#
# به‌جای آستانه روی شباهت خام، پرسش را رقابتی می‌کنیم:
#
#     prompts = ["a photo of a zebra",                    ← فرضیه
#                "a photo of a different kind of animal", ← رقیب
#                "a photo of scenery with no animal"]     ← رقیب
#
# چرا؟ شباهت کسینوسی «فرضیه‌ی صفر» ندارد — هر برشی یک عددی می‌دهد و جای
# برش اصولی ندارد. رقیب‌ها به مدل اجازه می‌دهند بگوید «هیچ‌کدام».

import math
from dataclasses import dataclass


def _softmax(values):
    top = max(values)
    exps = [math.exp(v - top) for v in values]      # پایدار عددی
    total = sum(exps)
    return [e / total for e in exps]


def _sigmoid(v):
    if v >= 0:
        return 1.0 / (1.0 + math.exp(-v))
    e = math.exp(v)
    return e / (1.0 + e)


def hypothesis_score(logits, mode=None):
    """امتیاز فرضیه (اولین prompt) در برابر رقبا.

    ⚠️ نکته‌ی حیاتی: CLIP و SigLIP قابل تعویض نیستند.

       CLIP   با softmax روی یک دسته آموزش دیده → لاجیت‌ها فقط «نسبت به
              هم» معنی دارند و جمعشان همیشه ۱ است.
       SigLIP با سیگموید مستقل برای هر جفت → لاجیت یک bias آموخته‌شده
              دارد که عدد را «مطلق» می‌کند.

       اگر روی SigLIP سافت‌مکس بزنی، آن bias حذف می‌شود و همه‌ی آستانه‌ها
       بی‌صدا خراب می‌شوند: کد کار می‌کند، خطا نمی‌دهد، فقط جواب‌ها غلط‌اند.
    """
    mode = mode or SCORE_MODE
    if not logits:
        raise ValueError("logits خالی")
    if len(logits) == 1:
        raise ValueError("حداقل یک prompt رقیب لازم است؛ با یک کاندیدا نتیجه بی‌معنی است")

    if mode == "softmax":
        return _softmax(list(logits))[0]

    if mode == "sigmoid":
        probs = [_sigmoid(v) for v in logits]
        hyp, rivals = probs[0], probs[1:]
        best_rival = max(rivals)
        if best_rival >= hyp:
            # رقیب دست‌کم به همان خوبی توضیح می‌دهد → سرکوبش کن
            return hyp * (hyp / (hyp + best_rival))
        return hyp

    raise ValueError("حالت امتیازدهی ناشناخته: " + str(mode))


def gate_score(logits, n_positive, mode=None):
    """اختلاف بهترین prompt مثبت با بهترین منفی. بازه [-1, 1]."""
    mode = mode or SCORE_MODE
    if n_positive <= 0 or n_positive >= len(logits):
        raise ValueError("حداقل یک prompt مثبت و یک منفی لازم است")
    probs = [_sigmoid(v) for v in logits] if mode == "sigmoid" else _softmax(list(logits))
    return max(probs[:n_positive]) - max(probs[n_positive:])


contrastive_score = hypothesis_score        # سازگاری با نام قدیمی


@dataclass(frozen=True)
class Match:
    file_name: str
    file_path: str
    score: float
    box: tuple = None
    label: str = None


def best_per_image(matches):
    """یک نتیجه به‌ازای هر تصویر — قوی‌ترین نمونه.

    🐞 نسخه‌ی یکپارچه break هر تصویر را از دست داده بود، پس عکسی با دو
       چهره‌ی منطبق دو بار ظاهر می‌شد. ضمناً «اولین» نمونه ثبت می‌شد و
       همان کلید مرتب‌سازی بود — پس عکسی که گربه‌ی دومش ۹۵٪ بود، با ۲۶٪
       رتبه‌بندی می‌شد.
    """
    best = {}
    for m in matches:
        cur = best.get(m.file_name)
        if cur is None or m.score > cur.score:
            best[m.file_name] = m
    return sorted(best.values(), key=lambda m: m.score, reverse=True)


assert abs(sum(_softmax([1.0, 2.0, 3.0])) - 1.0) < 1e-9
assert hypothesis_score([10.0, 0.0, 0.0], "softmax") > 0.95
assert hypothesis_score([0.0, 10.0, 0.0], "sigmoid") < 0.2
print("✅ امتیازدهی رقابتی آماده")

In [ ]:
# ── دروازه‌ی کیفیت چهره + نواحی چندبرشی ─────────────────────────────

def is_usable_face(det_score, box, min_det_score=0.30, min_px=24):
    """آیا این چهره ارزش امبدینگ‌گرفتن دارد؟

    ⚠️ عمداً *ملایم* است. هدف انتخابی‌بودن نیست — هر چیزی که RetinaFace
       کمی به آن مطمئن باشد ایندکس می‌شود.

       دلیل وجودش: امبدینگِ چهره‌ی ۱۲ پیکسلیِ محو، «سیگنال ضعیف» نیست —
       *گمراه‌کننده* است. در آستانه‌ی ۰.۴۶ چنین امبدینگی صرفاً بی‌جواب
       نمی‌ماند، بلکه با **فرد اشتباه** تطابق می‌دهد.
    """
    if det_score < min_det_score:
        return False
    x1, y1, x2, y2 = box
    return min(x2 - x1, y2 - y1) >= min_px


def crop_regions(width, height, layout="1+2x2+center"):
    """نواحی‌ای که برای هر عکس امبد می‌شوند.

    یک امبدینگ از کل تصویر، کل صحنه را میانگین می‌گیرد. کوله‌پشتی‌ای که
    ۳٪ کادر است تقریباً هیچ اثری روی بردار ندارد — همان دلیلی که دموی
    اولیه برای بهترین نتیجه‌اش ۲۳٪ می‌داد.

    برش مرکزی مهم است: شبکه‌ی ۲×۲ دقیقاً از وسط کادر رد می‌شود، یعنی
    جایی که عکاس سوژه را می‌گذارد — بدون آن، سوژه‌ی وسط بین چهار ربع
    تکه می‌شود و در هیچ‌کدام کامل نیست.
    """
    if width <= 0 or height <= 0:
        raise ValueError("ابعاد نامعتبر")

    regions = [("whole", (0, 0, width, height))]
    if layout == "1":
        return regions

    if "2x2" in layout:
        mx, my = width // 2, height // 2
        regions += [("tl", (0, 0, mx, my)), ("tr", (mx, 0, width, my)),
                    ("bl", (0, my, mx, height)), ("br", (mx, my, width, height))]

    if "center" in layout:
        qw, qh = width // 4, height // 4
        regions.append(("center", (qw, qh, width - qw, height - qh)))

    return regions


CROP_LAYOUT = "1+2x2+center"        # ۶ بردار به‌ازای هر عکس

assert is_usable_face(0.35, (0, 0, 30, 34))       # ملایم: قبول
assert not is_usable_face(0.99, (0, 0, 12, 14))   # خیلی ریز: رد
print("✅ کیفیت چهره +", len(crop_regions(1000, 1000, CROP_LAYOUT)), "ناحیه در هر عکس")

## ⚙️ تنظیمات

همه‌ی ثابت‌ها یکجا — نه پخش‌شده در کد.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class FaceConfig:
    gallery: str = "/content/gallery"
    database: str = "face_index.db"

    match_threshold: float = 0.46      # کسینوس روی امبدینگ ArcFace

    det_size_large: int = 640
    det_size_small: int = 320
    small_image_px: int = 400

    # سقف بُعد هنگام ایندکس: عکس خبری ۶۰۰۰ پیکسلی زمان دیکد و
    # پهنای باند می‌خورد برای جزئیاتی که مدل ۶۴۰ هرگز نمی‌بیند
    max_index_dimension: int = 1600

    workers: int = 8                   # نخ‌های لود موازی
    checkpoint_every: int = 50         # هر چند تصویر commit شود

    # ── دروازه‌ی کیفیت (جدید) ──
    # عمداً ملایم: هدف انتخابی‌بودن نیست، فقط حذف چهره‌هایی که اصلاً
    # نمی‌توانند هویت را حمل کنند. امبدینگ چهره‌ی ۱۲ پیکسلیِ محو
    # «سیگنال ضعیف» نیست — با فرد اشتباه تطابق می‌دهد.
    min_det_score: float = 0.30
    min_face_px: int = 24

CFG = FaceConfig()
print(CFG)

## 🧠 بارگذاری مدل

In [ ]:
from insightface.app import FaceAnalysis

providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
             if RT.is_cuda else ["CPUExecutionProvider"])

face_app = FaceAnalysis(name="buffalo_l", providers=providers)
face_app.prepare(ctx_id=0 if RT.is_cuda else -1,
                 det_size=(CFG.det_size_large, CFG.det_size_large))
print("✅ InsightFace buffalo_l مستقر شد")

## 🗄️ پایگاه داده

**✅ رفع باگ ۱** — کلید اصلی `(file_name, bbox)` کلید طبیعی این جدول است:
ایندکس مجدد idempotent می‌شود، ولی دو چهره‌ی متفاوت در یک عکس هر دو
نگه داشته می‌شوند.

`content_hash` هم اضافه شد تا عکس **ویرایش‌شده** دوباره ایندکس شود —
قبلاً فقط نام فایل ملاک بود.

In [ ]:
import hashlib
import sqlite3
from contextlib import contextmanager
from pathlib import Path

SCHEMA = """
CREATE TABLE IF NOT EXISTS gallery_meta (
    file_name    TEXT PRIMARY KEY,
    file_path    TEXT NOT NULL,
    content_hash TEXT NOT NULL,
    has_face     INTEGER NOT NULL DEFAULT 0
);

-- کلید اصلی (file_name, bbox) = کلید طبیعی: یک ردیف به‌ازای هر چهره
CREATE TABLE IF NOT EXISTS face_embeddings (
    file_name TEXT NOT NULL,
    bbox      TEXT NOT NULL,
    embedding BLOB NOT NULL,
    PRIMARY KEY (file_name, bbox),
    FOREIGN KEY (file_name) REFERENCES gallery_meta(file_name) ON DELETE CASCADE
);

CREATE INDEX IF NOT EXISTS idx_has_face ON gallery_meta(has_face);
"""

@contextmanager
def connect(db_path=None):
    conn = sqlite3.connect(db_path or CFG.database)
    try:
        conn.execute("PRAGMA foreign_keys = ON")
        yield conn
        conn.commit()
    finally:
        conn.close()

def init_db():
    with connect() as conn:
        conn.executescript(SCHEMA)
    print("پایگاه داده آماده:", CFG.database)

def content_hash(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

init_db()

## 📥 ایندکس‌گذاری

**✅ رفع باگ ۴** — سه چیزی که نسخه یکپارچه از دست داده بود برگشت:

1. **ThreadPoolExecutor** — دیکد تصویر با استنتاج GPU همپوشانی می‌کند
   به‌جای اینکه پشت آن صف بکشد
2. **پیش‌رسایز** — سقف بُعد قبل از تشخیص
3. **Checkpoint** — هر ۵۰ تصویر commit، پس قطع‌شدن کل کار را نمی‌سوزاند

ضمناً کشف فایل **به حروف بزرگ/کوچک حساس نیست**: نسخه قبلی فقط الگوهای
کوچک را glob می‌کرد، پس فایل‌های `.JPG` (پیش‌فرض اکثر دوربین‌ها) بی‌صدا
نامرئی بودند.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import cv2
from tqdm.auto import tqdm

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

def discover_images(root):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError("گالری پیدا نشد: " + str(root))
    return sorted(p for p in root.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)

def load_and_resize(path):
    """کارگر نخ: لود از دیسک + کوچک‌سازی هوشمند."""
    try:
        img = cv2.imread(str(path))
        if img is None:
            return path, None
        h, w = img.shape[:2]
        if max(h, w) > CFG.max_index_dimension:
            s = CFG.max_index_dimension / max(h, w)
            img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
        return path, img
    except Exception:
        return path, None

def build_face_index():
    paths = discover_images(CFG.gallery)
    print(len(paths), "تصویر در گالری")
    stats = {"indexed": 0, "skipped": 0, "failed": 0}

    with connect() as conn:
        pending = []
        for p in paths:
            h = content_hash(p)
            row = conn.execute(
                "SELECT content_hash FROM gallery_meta WHERE file_name = ?", (p.name,)
            ).fetchone()
            if row is None or row[0] != h:
                pending.append((p, h))
            else:
                stats["skipped"] += 1

        if not pending:
            print("ایندکس به‌روز است —", stats)
            return stats

        print("ایندکس", len(pending), "تصویر جدید یا تغییریافته")
        hashes = dict(pending)

        with ThreadPoolExecutor(max_workers=CFG.workers) as pool:
            loaded = pool.map(load_and_resize, [p for p, _ in pending])

            for path, img in tqdm(loaded, total=len(pending), desc="Indexing"):
                if img is None:
                    stats["failed"] += 1
                    continue
                try:
                    # ❌ قبلاً: هر چهره‌ای که پیدا می‌شد ایندکس می‌شد
                    #    faces = face_app.get(img)
                    # ✅ حالا: فیلتر ملایم کیفیت
                    faces = [f for f in face_app.get(img)
                             if is_usable_face(float(f.det_score),
                                               tuple(int(v) for v in f.bbox[:4]),
                                               CFG.min_det_score, CFG.min_face_px)]

                    # پاک‌کردن ردیف‌های قبلی این فایل (برای حالت ویرایش)
                    conn.execute("DELETE FROM face_embeddings WHERE file_name = ?", (path.name,))
                    conn.execute(
                        "INSERT INTO gallery_meta (file_name, file_path, content_hash, has_face) "
                        "VALUES (?,?,?,?) ON CONFLICT(file_name) DO UPDATE SET "
                        "file_path=excluded.file_path, content_hash=excluded.content_hash, "
                        "has_face=excluded.has_face",
                        (path.name, str(path), hashes[path], int(len(faces) > 0)),
                    )
                    for f in faces:
                        box = ",".join(str(int(v)) for v in f.bbox[:4])
                        conn.execute(
                            "INSERT OR REPLACE INTO face_embeddings VALUES (?,?,?)",
                            (path.name, box, to_blob(f.normed_embedding)),
                        )

                    stats["indexed"] += 1
                    if stats["indexed"] % CFG.checkpoint_every == 0:
                        conn.commit()          # checkpoint
                except Exception as e:
                    # قبلاً except خالی بود و خطا را می‌بلعید
                    print("خطا در", path.name, ":", e)
                    stats["failed"] += 1

    print("تمام شد —", stats)
    return stats

# build_face_index()

## 🔎 جستجو

**✅ رفع باگ ۳** — `best_per_image` یک نتیجه به‌ازای هر عکس برمی‌گرداند، و
**قوی‌ترین** چهره را نگه می‌دارد نه اولی. این دومی مهم است: کد قبلی اولین
نمونه را ثبت می‌کرد و همان را کلید مرتب‌سازی می‌کرد.

In [ ]:
import time

def get_query_embedding(img_path):
    """امبدینگ سوژه‌ی عکس پرس‌وجو (بزرگ‌ترین چهره)."""
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError("تصویر خوانده نشد: " + str(img_path))

    h, w = img.shape[:2]
    size = CFG.det_size_large if max(h, w) > CFG.small_image_px else CFG.det_size_small
    face_app.prepare(ctx_id=0 if RT.is_cuda else -1, det_size=(size, size))
    faces = face_app.get(img)

    if not faces:
        # تلاش مجدد با اندازه کامل: پرتره‌های کوچک یا تنگ‌برش‌خورده
        face_app.prepare(ctx_id=0 if RT.is_cuda else -1,
                         det_size=(CFG.det_size_large, CFG.det_size_large))
        faces = face_app.get(img)
    if not faces:
        raise ValueError("چهره‌ای در عکس پرس‌وجو پیدا نشد")

    # بزرگ‌ترین چهره سوژه است؛ بقیه رهگذرند
    largest = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    return largest.normed_embedding


def search_face(query_image, threshold=None):
    threshold = CFG.match_threshold if threshold is None else threshold
    query = np.asarray(get_query_embedding(query_image), dtype=np.float32).ravel()

    t0 = time.time()
    matches = []
    with connect() as conn:
        rows = conn.execute(
            "SELECT f.file_name, f.bbox, f.embedding, g.file_path "
            "FROM face_embeddings f JOIN gallery_meta g USING(file_name)"
        ).fetchall()

        for name, bbox, blob, path in rows:
            emb = from_blob(blob)
            if emb.shape != query.shape:
                # قبلاً بی‌صدا رد می‌شد — یعنی ایندکس با مدل دیگری ساخته شده
                raise ValueError("ناسازگاری ابعاد در " + name)
            sim = float(np.dot(query, emb))     # هر دو نرمال ⇒ کسینوس
            if sim >= threshold:
                matches.append(Match(name, path, sim,
                                     box=tuple(int(v) for v in bbox.split(","))))

    results = best_per_image(matches)
    print("اسکن", len(rows), "چهره در", round(time.time() - t0, 4), "ثانیه")
    print(len(results), "تصویر پیدا شد")
    for i, m in enumerate(results, 1):
        print(" [%2d] %-28s %6.2f%%" % (i, m.file_name, m.score * 100))
    return results

# results = search_face("/content/query.jpg")

## 📋 خلاصه‌ی این ماژول

| قبلاً | الان |
|:--|:--|
| بدون کلید اصلی → ردیف تکراری | `PRIMARY KEY (file_name, bbox)` |
| `pickle` — ریسک امنیتی + حجیم | `float32` خام |
| عکس دوچهره دو بار در نتایج | یک نتیجه، قوی‌ترین چهره |
| بدون ThreadPool و checkpoint | لود موازی ۸ نخ + commit هر ۵۰ |
| تصمیم فقط بر اساس نام فایل | هش محتوا → ویرایش تشخیص داده می‌شود |
| `.JPG` نامرئی | کشف بدون حساسیت به حروف |
| `except:` خالی | خطا چاپ و شمارش می‌شود |